<a href="https://colab.research.google.com/github/giuliadesantis/Project_IS_Group43/blob/main/STEP8_CocoFineTuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 8 - Anomaly Segmentation EoMT-Coco Fine Tuned

# Repository, Drive, Requirements and import SetUp

In [1]:
!git clone https://github.com/giuliadesantis/Project_IS_Group43.git
%cd Project_IS_Group43/eomt

fatal: destination path 'Project_IS_Group43' already exists and is not an empty directory.
/content/Project_IS_Group43/eomt


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
sys.path.append('/content/Project_IS_Group43')
sys.path.append('/content/Project_IS_Group43/eval')
sys.path.append('/content/Project_IS_Group43/eomt')


In [5]:
!pip uninstall -y torch torchvision torchaudioa
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install ood-metrics lightning
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchvision==/d' requirements.txt
!python3 -m pip install -r requirements.txt

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 60.6 MB/s

In [2]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

In [3]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib
import gc

# Setup

In [4]:
seed_everything(0, verbose=False) # reproducible results

device = 0  # type of torch device
with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

# Load model

In [5]:
current_config = config_cs # we use the confgiguration of cityscapes beacuse we have performed the fine-tuning

# we retrieve the checkpoint of the finetuned model
path_checkpoint = "/content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last-v2.ckpt"
path_bin_output = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco_finetuned.bin" # path of the converted ckpt in bin
checkpoint = torch.load(path_checkpoint, map_location="cpu", weights_only=False)
if "state_dict" in checkpoint:
    state_dict_clean = checkpoint["state_dict"]
else:
    state_dict_clean = checkpoint
torch.save(state_dict_clean, path_bin_output) # save the bin of the EoMT Fine tuned model

target_img_size = (640, 640)
num_classes_to_load = 19
state_dict_path = path_bin_output
num_queries_to_load = 200 # num queries expected from COCO since Cityscapes' config has 100

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network_kwargs["num_q"] = num_queries_to_load # fix the num queries to the one expected (200)
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

# Load Pretrained

In [ ]:
model_kwargs_final = {k: v for k, v in model_kwargs.items()}

if "num_classes" in model_kwargs_final:
  del model_kwargs_final["num_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True)
    model.load_state_dict(state_dict, strict=False)

# Evaluation Anomaly Metrics

In [ ]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        #ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)

In [ ]:
parser = ArgumentParser()

parser.add_argument(
    "--input",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg", # change the default path based on the dataset folder
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp",
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)

parser.add_argument('--num-workers', type=int, default=2)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
parser.add_argument('--device', type=str, default='cuda', help="cpu or cuda, the device used for evaluation")
args = parser.parse_args(args=[])

# initialize anomaly scores lists
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []


In [ ]:
# Method to transform the mask prediction of EoMT. We obtain the logits of the pixels.
def get_eomt_logits(model, img_tensor, target_size):
    model.eval()
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img_tensor.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        crops, origins = model.window_imgs_semantic(imgs) # handle crops of EoMT

        mask_logits_per_layer, class_logits_per_layer = model(crops) # masks prediction

        mask_logits = F.interpolate( # Upsampling of masks
            mask_logits_per_layer[-1], target_size, mode="bilinear")

        crop_logits = model.to_per_pixel_logits_semantic( # get the logits
            mask_logits, class_logits_per_layer[-1])

        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes) # reconstruct the image

    return logits

In [ ]:
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

# We create a folder to save logits (this is useful for the next temperature scaling step)
#os.makedirs("saved_logits_coco_RoadAnomaly", exist_ok=True) # change the path based on the anomaly dataset
#os.makedirs("saved_logits_coco_RoadAnomaly21", exist_ok=True)
#os.makedirs("saved_logits_coco_fs_static", exist_ok=True)
#os.makedirs("saved_logits_coco_FS_LostFound_full", exist_ok=True)
os.makedirs("saved_logits_coco_RoadObsticle21", exist_ok=True)

for path in glob.glob(os.path.expanduser(str(input_pattern))):

    img_pil = input_transform((Image.open(path).convert('RGB')))
    images = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)

    logits_list = get_eomt_logits(model, images, target_img_size) # get the logits predictions, it returns a list
    logits = logits_list[0].unsqueeze(0)

    nome_file_logit = os.path.basename(path).replace(".jpg", ".pt").replace(".png", ".pt").replace(".webp", ".pt")

    # we save the logits in the folder we have created
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly", nome_file_logit)) # da cambiare in base al dataset
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly21", nome_file_logit))
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_fs_static", nome_file_logit))
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_FS_LostFound_full", nome_file_logit))
    torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadObsticle21", nome_file_logit))

    # Max Logit
    anomaly_result_maxlogit = 1.0 - torch.max(logits.squeeze(0), dim=0)[0].cpu().numpy()

    # MSP
    probs = torch.nn.functional.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    # Max Entropy
    probs = probs.squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    # RbA
    anomaly_result_rba = -torch.sum(torch.tanh(logits.squeeze(0)), dim=0).cpu().numpy()

    # handle images' formats
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
        # in RoadAnomaly dataset 2 means anomaly. We transform it to standardize wrt other datasets where anomaly is indicated with 1
        ood_gts = np.where((ood_gts==2), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue   # if there is no anomaly pixel in the image, skip the image and go to the next
    else:
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)
          anomaly_score_list_rba.append(anomaly_result_rba)

    # to avoid outOfMemory errors
    del logits_list, logits, probs
    del anomaly_result_maxlogit, anomaly_result_msp, anomaly_result_maxentropy, anomaly_result_rba
    del ood_gts, mask, img_pil, images
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# compute the final anomaly detection metrics: AuPRC and FPR95

ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)
anomaly_scores_rba = np.array(anomaly_score_list_rba)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# Max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AuPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')


# MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')


# Max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')


# RbA
ood_out = anomaly_scores_rba[ood_mask]
ind_out = anomaly_scores_rba[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score rba: {prc_auc*100.0}')
print(f'FPR@TPR95 rba: {fpr*100.0}')

AuPRC score maxlogit: 74.2955474673972
FPR@TPR95 maxlogit: 4.521412826746512
AUPRC score msp: 74.79343517455996
FPR@TPR95 msp: 2.3476107592840396
AUPRC score maxentropy: 73.44034502056218
FPR@TPR95 maxentropy: 23.84307224526027
AUPRC score rba: 63.25599084047301
FPR@TPR95 rba: 94.02226115873289


# Temperature scaling


In [ ]:
# we retrieve the saved logits to avoid recomputing them
#logits_folder = "saved_logits_coco_RoadAnomaly" # change based on the dataset we use
#logits_folder = "saved_logits_coco_RoadAnomaly21"
#logits_folder = "saved_logits_coco_fs_static"
#logits_folder = "saved_logits_coco_FS_LostFound_full"
logits_folder = "saved_logits_coco_RoadObsticle21"

#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/labels_masks" # change based on the dataset we use
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/labels_masks"
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/labels_masks"
#masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/labels_masks"
masks_folder = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/labels_masks"

# Emprirical grid search over a set of different temperatures
# T=1.0 is the baseline with MSP
temperatures = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 5.0, 10.0, 100.0]

for T in temperatures:
    score_msp_T = []
    gts_T = []

    for logit_path in glob.glob(os.path.join(logits_folder, "*.pt")):

        logits = torch.load(logit_path)
        probs = torch.nn.functional.softmax(logits / T, dim=1)
        max_probs, _ = torch.max(probs.squeeze(0), dim=0)
        anomaly_msp = 1.0 - max_probs.numpy()

        base_name = os.path.basename(logit_path).replace(".pt", "")
        pathGT = os.path.join(masks_folder, f"{base_name}.png")

        if not os.path.exists(pathGT):
            continue

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        if "RoadAnomaly" in masks_folder:
            ood_gts = np.where((ood_gts==2), 1, ood_gts)

        if 1 in np.unique(ood_gts):
            gts_T.append(ood_gts)
            score_msp_T.append(anomaly_msp)

    if len(gts_T) > 0:
        val_gts = np.concatenate(gts_T)
        val_msp = np.concatenate(score_msp_T)

        ood_mask = (val_gts == 1)
        ind_mask = (val_gts == 0)

        val_out_msp = np.concatenate((val_msp[ind_mask], val_msp[ood_mask]))
        val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

        auprc_msp = average_precision_score(val_label, val_out_msp)
        fpr_msp = fpr_at_95_tpr(val_out_msp, val_label)

        print(f"=== Temperature T = {T} ===")
        print(f"AUPRC MSP: {auprc_msp*100:.2f}%  |  FPR95 MSP: {fpr_msp*100:.2f}%\n")
    else:
        print(f"Warning: No anomaly mask for T={T}")

=== Temperature T = 0.01 ===
AUPRC MSP: 27.90%  |  FPR95 MSP: 89.04%

=== Temperature T = 0.05 ===
AUPRC MSP: 54.08%  |  FPR95 MSP: 37.22%

=== Temperature T = 0.1 ===
AUPRC MSP: 69.95%  |  FPR95 MSP: 34.06%

=== Temperature T = 0.2 ===
AUPRC MSP: 73.00%  |  FPR95 MSP: 19.01%

=== Temperature T = 0.5 ===
AUPRC MSP: 74.67%  |  FPR95 MSP: 3.14%

=== Temperature T = 1.0 ===
AUPRC MSP: 74.79%  |  FPR95 MSP: 2.35%

=== Temperature T = 1.5 ===
AUPRC MSP: 74.78%  |  FPR95 MSP: 2.33%

=== Temperature T = 2.0 ===
AUPRC MSP: 74.77%  |  FPR95 MSP: 2.33%

=== Temperature T = 5.0 ===
AUPRC MSP: 74.73%  |  FPR95 MSP: 2.35%

=== Temperature T = 10.0 ===
AUPRC MSP: 74.72%  |  FPR95 MSP: 2.36%

=== Temperature T = 100.0 ===
AUPRC MSP: 74.70%  |  FPR95 MSP: 2.36%

